# Step 05 — POI-to-Building Match & Condensed Dataset

**Input:**
- `data/output/04_enriched_building_volume_data.gpkg`
- `data/output/03_osm_pois_modified.gpkg`
- `data/output/01_all_buildings_osm.gpkg` (for gap-filling)

**Output:** `data/output/05_condensed_buildings_with_pois.gpkg`

**Steps:**
1. Remove non-enterable building types from ALKIS data
2. Apply POI exclusion filters (re-enabled: parking, benches, sheds, etc.)
3. Add OSM-only buildings missing from ALKIS (new construction)
4. Spatial join: POIs → buildings (intersect, then nearest fallback ≤100 m)
5. Consolidate: one row per building with merged POI info
6. Filter: volume ≥ MIN_CONDENSED_VOLUME_M3

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Config loaded')

## 1. Load enriched buildings and remove non-enterable types

In [ ]:
gdf = gpd.read_file(ENRICHED_BUILDINGS_FILE)
print(f'Loaded {len(gdf):,} enriched buildings')

# Drop address columns into single field before schema reduction
addr_parts = ['Name', 'Strasse', 'HausNr', 'Stadt']
available  = [c for c in addr_parts if c in gdf.columns]
gdf['alkis_address'] = gdf[available].apply(
    lambda row: ' '.join([str(v) for v in row if pd.notna(v) and str(v).strip()]), axis=1
)
gdf = gdf.drop(columns=available, errors='ignore')

# Remove non-enterable building types
gdf = gdf[~gdf['label_en'].apply(
    lambda x: any(label in LABELS_TO_REMOVE for label in (x if isinstance(x, list) else [x]))
)].copy()
print(f'After removing non-enterable types: {len(gdf):,}')

# Keep only needed columns
final_cols = ['gml_id', 'volume_m3', 'label_en', 'osm_names', 'osm_building_type',
              'osm_landuse_class', 'osm_landuse_name', 'gfk_class',
              'ALKIS_Landuse_info', 'alkis_address', 'geometry']
final_cols = [c for c in final_cols if c in gdf.columns]
gdf_base = gdf[final_cols].to_crs(TARGET_CRS).copy()
gdf_base['geometry'] = gdf_base.geometry.buffer(0)

## 2. Add OSM-only buildings (gap-fill for new construction)

In [ ]:
if ALL_BUILDINGS_OSM_FILE.exists():
    osm_fill = gpd.read_file(ALL_BUILDINGS_OSM_FILE, layer='buildings').to_crs(TARGET_CRS)
    osm_fill = osm_fill[osm_fill.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    osm_fill = osm_fill[osm_fill['building'].notna()].copy()
    osm_fill['geometry'] = osm_fill.geometry.buffer(0)
    osm_fill = osm_fill[osm_fill.geometry.notna() & ~osm_fill.geometry.is_empty].copy()

    join_check  = gpd.sjoin(osm_fill, gdf_base[['geometry']], how='left', predicate='intersects')
    osm_missing = osm_fill.loc[join_check[join_check['index_right'].isna()].index].copy()

    def to_float_safe(x):
        try: return float(str(x).replace(',', '.'))
        except: return np.nan

    def infer_height(row):
        h = to_float_safe(row.get('height'))
        if pd.notna(h): return h
        lvl = to_float_safe(row.get('building:levels'))
        if pd.notna(lvl): return lvl * DEFAULT_FLOOR_HEIGHT_M
        return DEFAULT_FLOORS * DEFAULT_FLOOR_HEIGHT_M

    volume_m3 = osm_missing.geometry.area * osm_missing.apply(infer_height, axis=1)

    osm_to_add = gpd.GeoDataFrame({
        'gml_id':            'osm_' + osm_missing['id'].astype(str),
        'volume_m3':         volume_m3.values,
        'label_en':          pd.NA,
        'osm_names':         osm_missing['name'].values if 'name' in osm_missing else np.nan,
        'osm_building_type': osm_missing['building'].values,
        'osm_landuse_class': pd.NA,
        'osm_landuse_name':  pd.NA,
        'gfk_class':         pd.NA,
        'ALKIS_Landuse_info': pd.NA,
        'alkis_address':     pd.NA,
        'geometry':          osm_missing.geometry.values,
    }, crs=TARGET_CRS)

    osm_to_add = osm_to_add[[c for c in final_cols if c in osm_to_add.columns]]
    gdf_base   = gdf_base[[c for c in final_cols if c in gdf_base.columns]]

    gdf = pd.concat([gdf_base, osm_to_add], ignore_index=True)
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=TARGET_CRS)
    print(f'ALKIS buildings: {len(gdf_base):,}  +  OSM-only: {len(osm_to_add):,}  =  {len(gdf):,} total')
else:
    gdf = gdf_base.copy()
    print('OSM buildings file not found — using ALKIS only')

## 3. Load and filter POIs

**Bug fix: amenity and building exclusion filters are now active.**

In [ ]:
clean_pois = gpd.read_file(OSM_POIS_MODIFIED_FILE)
print(f'Loaded {len(clean_pois):,} POIs')

# Remove non-building POI types (was commented out — now re-enabled)
before = len(clean_pois)
if 'amenity' in clean_pois.columns:
    clean_pois = clean_pois[~clean_pois['amenity'].isin(EXCLUDE_AMENITIES)].copy()
if 'building' in clean_pois.columns:
    clean_pois = clean_pois[~clean_pois['building'].isin(EXCLUDE_BUILDING_TYPES)].copy()
if 'tourism' in clean_pois.columns:
    clean_pois = clean_pois[
        clean_pois['tourism'].isna() | clean_pois['tourism'].isin(ALLOWED_TOURISM_TYPES)
    ].copy()
if 'information' in clean_pois.columns:
    clean_pois = clean_pois[
        clean_pois['information'].isna() | clean_pois['information'].isin(ALLOWED_INFORMATION_TYPES)
    ].copy()

print(f'After exclusion filters: {len(clean_pois):,} (removed {before - len(clean_pois):,})')

# Convert polygon POIs to representative points
clean_pois = clean_pois[~clean_pois.geom_type.isin(['LineString', 'MultiLineString'])].copy()
clean_pois = clean_pois.to_crs(TARGET_CRS)
poly_mask  = clean_pois.geom_type.isin(['Polygon', 'MultiPolygon'])
clean_pois.loc[poly_mask, 'geometry'] = clean_pois.loc[poly_mask, 'geometry'].representative_point()

if gdf.crs != clean_pois.crs:
    clean_pois = clean_pois.to_crs(gdf.crs)

## 4. Spatial join: POIs → buildings (intersect + nearest fallback)

In [ ]:
def clean_join_cols(df):
    return df.drop(columns=['index_right', 'index_left'], errors='ignore')

gdf = clean_join_cols(gdf.to_crs(TARGET_CRS).copy())
clean_pois = clean_join_cols(clean_pois.to_crs(TARGET_CRS).copy())
gdf['gml_id'] = gdf.index  # use positional index as join key

pois_joined = gpd.sjoin(clean_join_cols(clean_pois), gdf[['gml_id', 'geometry']],
                        how='left', predicate='intersects')
pois_joined['match_type'] = pois_joined['gml_id'].notna().map({True: 'intersects', False: None})

matched            = pois_joined[pois_joined['gml_id'].notna()].copy()
unmatched_initial  = pois_joined[pois_joined['gml_id'].isna()].copy()

if not unmatched_initial.empty:
    nearest = gpd.sjoin_nearest(
        clean_join_cols(unmatched_initial.drop(columns=['gml_id'], errors='ignore')),
        gdf[['gml_id', 'geometry']],
        how='left', max_distance=100, distance_col='snap_dist'
    )
    nearest['match_type'] = nearest['gml_id'].notna().map({True: 'nearest', False: 'unmatched'})
    pois_final = pd.concat([matched, nearest], ignore_index=True)
else:
    pois_final = matched.copy()

print(pois_final['match_type'].value_counts(dropna=False))

result = gdf.merge(pois_final.drop(columns='geometry'), on='gml_id', how='left')
result = result.drop(columns=['index_right', 'match_type', 'snap_dist', 'id'], errors='ignore')

## 5. Consolidate: one row per building

In [ ]:
def is_missing(x):
    if x is None: return True
    try: return pd.isna(x)
    except: return False

def flatten_unique(series):
    out, seen = [], set()
    for v in series:
        if is_missing(v): continue
        items = v if isinstance(v, (list, tuple, set)) else [v]
        for item in items:
            key = str(item).strip()
            if key and key not in seen:
                seen.add(key); out.append(item)
    return out if out else None

def first_non_null(series):
    return next((v for v in series if not is_missing(v)), None)

def merge_names(osm_names_series, name_series):
    vals = []
    for v in osm_names_series:
        if is_missing(v): continue
        vals.extend([str(x).strip() for x in (v if isinstance(v, (list, tuple, set)) else [v])])
    for v in name_series:
        if not is_missing(v): vals.append(str(v).strip())
    seen, out = set(), []
    for v in vals:
        if v and v not in seen: seen.add(v); out.append(v)
    return out if out else None

single_value_cols = ['volume_m3', 'label_en', 'osm_building_type', 'osm_landuse_class',
                     'osm_landuse_name', 'gfk_class', 'ALKIS_Landuse_info', 'geometry']
list_cols         = ['email', 'amenity', 'building', 'shop', 'tourism', 'information',
                     'tags_search', 'additional_information']

rows = []
for gml_id, grp in result.groupby('gml_id', dropna=False):
    row = {'gml_id': gml_id}
    for col in single_value_cols:
        if col in grp.columns: row[col] = first_non_null(grp[col])
    row['osm_names']      = merge_names(grp.get('osm_names', pd.Series(dtype=object)),
                                        grp.get('name',      pd.Series(dtype=object)))
    row['alkis_address']  = first_non_null(grp.get('alkis_address', pd.Series(dtype=object)))
    for col in list_cols:
        if col in grp.columns: row[col] = flatten_unique(grp[col])
    if 'website' in grp.columns: row['website'] = flatten_unique(grp['website'])
    rows.append(row)

condensed = gpd.GeoDataFrame(rows, geometry='geometry', crs=result.crs)
print(f'Condensed: {len(condensed):,} unique buildings (was {len(result):,} rows)')

## 6. Apply volume filter and save

In [ ]:
condensed = condensed.drop(columns=['gfk_name'], errors='ignore')
condensed = condensed[condensed['volume_m3'].fillna(0) >= MIN_CONDENSED_VOLUME_M3]
print(f'After volume filter (≥ {MIN_CONDENSED_VOLUME_M3} m³): {len(condensed):,}')

condensed.to_file(CONDENSED_BUILDINGS_FILE, driver='GPKG')
print(f'Saved → {CONDENSED_BUILDINGS_FILE}')